# Tahap 3 — Validasi Kalender

**Tujuan notebook**
Membentuk *master calendar* harian dari 2017-01-01 sampai 2024-12-31, kemudian memetakan
ketersediaan data Ogimet dan SounderPy pada setiap tanggal kalender.

**Batasan cakupan (scope) — Tahap 3 TIDAK mencakup:**
- merge dataset
- pemilihan sounding harian / prioritas 12Z
- imputasi atau interpolasi
- penanganan outlier
- labeling
- feature engineering

Seluruh proses di atas adalah tanggung jawab tahap-tahap berikutnya (Tahap 4 dan seterusnya)
sesuai `implementation_playbook_dataset_lstm.md`.

**Deliverable utama:** `02_calendar_validation.csv`


## 1. Import Library

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


## 2. Konfigurasi

Konstanta path input/output dan batas master calendar. Sesuaikan `INPUT_DIR` bila lokasi
berkas berbeda dari direktori kerja notebook ini.

In [2]:
INPUT_DIR = Path(".")
OGIMET_PATH = INPUT_DIR / "ogimet_standardized.csv"
SOUNDERPY_PATH = INPUT_DIR / "sounderpy_standardized.csv"

OUTPUT_PATH = Path("02_calendar_validation.csv")

CALENDAR_START = "2017-01-01"
CALENDAR_END = "2024-12-31"

# Kolom kalender resmi SounderPy adalah "date" (identik 100% dengan nominal_date
# berdasarkan audit Tahap 2). observation_datetime TIDAK digunakan sebagai key kalender.
OGIMET_COLS = ["date"]
SOUNDERPY_COLS = ["date", "hour"]


## 3. Langkah 1–2: Load Data & Konversi Datetime

Hanya kolom yang diperlukan untuk validasi kalender yang dibaca (`usecols`), agar efisien
secara memori. Kolom `date` dikonversi menjadi `datetime64`.

In [3]:
def load_ogimet_dates(path: Path) -> pd.DataFrame:
    """Load hanya kolom 'date' dari Ogimet, dikonversi ke datetime."""
    df = pd.read_csv(path, usecols=OGIMET_COLS)
    df["date"] = pd.to_datetime(df["date"], errors="raise")
    return df


def load_sounderpy_dates(path: Path) -> pd.DataFrame:
    """Load hanya kolom 'date' dan 'hour' dari SounderPy, date dikonversi ke datetime."""
    df = pd.read_csv(path, usecols=SOUNDERPY_COLS)
    df["date"] = pd.to_datetime(df["date"], errors="raise")
    return df


In [4]:
ogimet_df = load_ogimet_dates(OGIMET_PATH)
sounderpy_df = load_sounderpy_dates(SOUNDERPY_PATH)

ogimet_df.head()


,date
0,2017-01-01
1,2017-01-02
2,2017-01-03
3,2017-01-04
4,2017-01-05


In [5]:
sounderpy_df.head()


,date,hour
0,2017-01-01,12
1,2017-01-02,12
2,2017-01-03,12
3,2017-01-04,12
4,2017-01-05,12


## 4. Langkah 3: Bangun Master Calendar

Master calendar mencakup seluruh tanggal harian dari `2017-01-01` sampai `2024-12-31`
tanpa celah (termasuk tahun kabisat).

In [6]:
def build_master_calendar(start: str, end: str) -> pd.DataFrame:
    """Bangun master calendar harian lengkap tanpa celah."""
    dates = pd.date_range(start=start, end=end, freq="D")
    return pd.DataFrame({"tanggal": dates})


master_calendar = build_master_calendar(CALENDAR_START, CALENDAR_END)
master_calendar.head()


,tanggal
0,2017-01-01
1,2017-01-02
2,2017-01-03
3,2017-01-04
4,2017-01-05


## 5. Langkah 4: Struktur Berbasis Tanggal Unik

Audit dan pemetaan ketersediaan dilakukan atas **himpunan tanggal unik**, bukan seluruh
record mentah.

In [7]:
def get_unique_dates(df: pd.DataFrame, col: str = "date") -> pd.Series:
    """Kembalikan himpunan tanggal unik (terurut) dari sebuah DataFrame."""
    return pd.Series(sorted(df[col].unique()))


ogimet_unique_dates = get_unique_dates(ogimet_df)
sounderpy_unique_dates = get_unique_dates(sounderpy_df)

ogimet_unique_set = set(ogimet_unique_dates)
sounderpy_unique_set = set(sounderpy_unique_dates)


## 6. Langkah 5: Audit Dasar

Menghitung jumlah record, jumlah tanggal unik, duplicate date, serta rentang tanggal
(min/max) pada masing-masing sumber.

In [8]:
def audit_source(df: pd.DataFrame, unique_dates: pd.Series, name: str) -> dict:
    """Hitung statistik audit dasar untuk satu sumber data."""
    n_records = len(df)
    n_unique_dates = len(unique_dates)
    n_duplicate_rows = int(df["date"].duplicated().sum())
    return {
        "sumber": name,
        "jumlah_record": n_records,
        "jumlah_tanggal_unik": n_unique_dates,
        "duplicate_date": n_duplicate_rows,
        "min_date": unique_dates.min().date(),
        "max_date": unique_dates.max().date(),
    }


audit_ogimet = audit_source(ogimet_df, ogimet_unique_dates, "Ogimet")
audit_sounderpy = audit_source(sounderpy_df, sounderpy_unique_dates, "SounderPy")

audit_summary = pd.DataFrame([audit_ogimet, audit_sounderpy])
audit_summary


,sumber,jumlah_record,jumlah_tanggal_unik,duplicate_date,min_date,max_date
0,Ogimet,3287,2923,364,2017-01-01,2024-12-31
1,SounderPy,2774,2774,0,2017-01-01,2024-12-16


**Catatan:** `duplicate_date` dihitung sebagai jumlah baris yang tanggalnya sudah
pernah muncul sebelumnya pada sumber yang sama (`Series.duplicated()`). Pada SounderPy,
duplicate date secara alami dapat terjadi karena satu tanggal dapat memiliki lebih dari
satu jam observasi (00Z dan/atau 12Z); hal ini murni dicatat pada Tahap 3 dan **tidak**
ditangani (dipilih/di-drop) di sini — penanganan seleksi jam menjadi tanggung jawab
Tahap 4.

## 7. Langkah 6: Status Ketersediaan per Tanggal

Untuk setiap tanggal pada master calendar, tentukan `status_ogimet` dan `status_sounderpy`
(`ADA` / `TIDAK`).

In [9]:
def map_status(master_calendar: pd.DataFrame, ogimet_set: set, sounderpy_set: set) -> pd.DataFrame:
    """Petakan status ketersediaan Ogimet dan SounderPy pada setiap tanggal kalender."""
    df = master_calendar.copy()
    df["status_ogimet"] = np.where(df["tanggal"].isin(ogimet_set), "ADA", "TIDAK")
    df["status_sounderpy"] = np.where(df["tanggal"].isin(sounderpy_set), "ADA", "TIDAK")
    return df


calendar_validation = map_status(master_calendar, ogimet_unique_set, sounderpy_unique_set)
calendar_validation.head()


,tanggal,status_ogimet,status_sounderpy
0,2017-01-01,ADA,ADA
1,2017-01-02,ADA,ADA
2,2017-01-03,ADA,ADA
3,2017-01-04,ADA,ADA
4,2017-01-05,ADA,ADA


## 8. Langkah 7: Jam Tersedia (dari kolom `hour`)

Untuk setiap tanggal, laporkan jam observasi SounderPy yang tersedia: `NONE`, `00Z`,
`12Z`, atau `00Z,12Z`. Langkah ini murni pelaporan ketersediaan — **tidak** melakukan
seleksi/prioritas sounding.

In [10]:
def map_jam_tersedia(sounderpy_df: pd.DataFrame) -> dict:
    """Bangun mapping tanggal -> label jam_tersedia berdasarkan kolom hour."""
    hour_label = {0: "00Z", 12: "12Z"}

    grouped = sounderpy_df.groupby("date")["hour"].apply(lambda s: sorted(set(s)))

    jam_map = {}
    for tanggal, hours in grouped.items():
        labels = [hour_label.get(h, f"{h:02d}Z") for h in hours]
        jam_map[tanggal] = ",".join(labels) if labels else "NONE"

    return jam_map


jam_tersedia_map = map_jam_tersedia(sounderpy_df)
calendar_validation["jam_tersedia"] = (
    calendar_validation["tanggal"].map(jam_tersedia_map).fillna("NONE")
)
calendar_validation.head()


,tanggal,status_ogimet,status_sounderpy,jam_tersedia
0,2017-01-01,ADA,ADA,12Z
1,2017-01-02,ADA,ADA,12Z
2,2017-01-03,ADA,ADA,12Z
3,2017-01-04,ADA,ADA,12Z
4,2017-01-05,ADA,ADA,12Z


## 9. Langkah 8: Kategori Ketersediaan

Klasifikasikan setiap tanggal ke salah satu dari empat kategori: `BOTH`,
`OGIMET_ONLY`, `SOUNDERPY_ONLY`, `NEITHER`.

In [11]:
def assign_kategori(df: pd.DataFrame) -> pd.Series:
    """Tentukan kategori ketersediaan berdasarkan status_ogimet dan status_sounderpy."""
    ada_ogimet = df["status_ogimet"] == "ADA"
    ada_sounderpy = df["status_sounderpy"] == "ADA"

    kondisi = [
        ada_ogimet & ada_sounderpy,
        ada_ogimet & ~ada_sounderpy,
        ~ada_ogimet & ada_sounderpy,
        ~ada_ogimet & ~ada_sounderpy,
    ]
    pilihan = ["BOTH", "OGIMET_ONLY", "SOUNDERPY_ONLY", "NEITHER"]
    return np.select(kondisi, pilihan, default="NEITHER")


calendar_validation["kategori"] = assign_kategori(calendar_validation)
calendar_validation.head()


,tanggal,status_ogimet,status_sounderpy,jam_tersedia,kategori
0,2017-01-01,ADA,ADA,12Z,BOTH
1,2017-01-02,ADA,ADA,12Z,BOTH
2,2017-01-03,ADA,ADA,12Z,BOTH
3,2017-01-04,ADA,ADA,12Z,BOTH
4,2017-01-05,ADA,ADA,12Z,BOTH


## 10. Langkah 9: Simpan Hasil

In [12]:
output_columns = ["tanggal", "status_ogimet", "status_sounderpy", "jam_tersedia", "kategori"]
calendar_validation[output_columns].to_csv(OUTPUT_PATH, index=False)

print(f"Tersimpan: {OUTPUT_PATH.resolve()}")


Tersimpan: /home/claude/work/02_calendar_validation.csv


## 11. Output Ringkas

Ringkasan audit kalender dan ketersediaan data. Contoh tanggal missing (jika ada)
ditampilkan maksimal 5 tanggal pertama dan 5 tanggal terakhir saja.

In [13]:
def print_summary(df: pd.DataFrame) -> None:
    """Cetak ringkasan hasil validasi kalender."""
    n_hari = len(df)
    kategori_counts = df["kategori"].value_counts()

    n_both = int(kategori_counts.get("BOTH", 0))
    n_ogimet_only = int(kategori_counts.get("OGIMET_ONLY", 0))
    n_sounderpy_only = int(kategori_counts.get("SOUNDERPY_ONLY", 0))
    n_neither = int(kategori_counts.get("NEITHER", 0))

    n_missing_ogimet = int((df["status_ogimet"] == "TIDAK").sum())
    n_missing_sounderpy = int((df["status_sounderpy"] == "TIDAK").sum())

    print(f"Jumlah hari kalender     : {n_hari}")
    print(f"Jumlah BOTH              : {n_both}")
    print(f"Jumlah OGIMET_ONLY       : {n_ogimet_only}")
    print(f"Jumlah SOUNDERPY_ONLY    : {n_sounderpy_only}")
    print(f"Jumlah NEITHER           : {n_neither}")
    print(f"Jumlah missing Ogimet    : {n_missing_ogimet}")
    print(f"Jumlah missing SounderPy : {n_missing_sounderpy}")


print_summary(calendar_validation)


Jumlah hari kalender     : 2922
Jumlah BOTH              : 2774
Jumlah OGIMET_ONLY       : 148
Jumlah SOUNDERPY_ONLY    : 0
Jumlah NEITHER           : 0
Jumlah missing Ogimet    : 0
Jumlah missing SounderPy : 148


In [14]:
def show_missing_examples(df: pd.DataFrame, status_col: str, label: str, n: int = 5) -> None:
    """Tampilkan maksimal n tanggal pertama dan n tanggal terakhir yang missing."""
    missing_dates = df.loc[df[status_col] == "TIDAK", "tanggal"]
    total = len(missing_dates)

    if total == 0:
        print(f"{label}: tidak ada tanggal missing.")
        return

    print(f"{label}: {total} tanggal missing (menampilkan maks {n} pertama & {n} terakhir)")
    print("  Pertama :", list(missing_dates.head(n).dt.date))
    print("  Terakhir:", list(missing_dates.tail(n).dt.date))


show_missing_examples(calendar_validation, "status_ogimet", "Missing Ogimet")
print()
show_missing_examples(calendar_validation, "status_sounderpy", "Missing SounderPy")


Missing Ogimet: tidak ada tanggal missing.

Missing SounderPy: 148 tanggal missing (menampilkan maks 5 pertama & 5 terakhir)
  Pertama : [datetime.date(2017, 6, 12), datetime.date(2017, 6, 13), datetime.date(2017, 6, 14), datetime.date(2017, 6, 15), datetime.date(2017, 8, 16)]
  Terakhir: [datetime.date(2024, 12, 27), datetime.date(2024, 12, 28), datetime.date(2024, 12, 29), datetime.date(2024, 12, 30), datetime.date(2024, 12, 31)]
